# Track 1 — LoRA Fine-Tuning Orchestrator

Thin orchestrator only. All real logic lives in `track1_finetune/scripts/*.py`
(agent-editable, ordinary `.py` modules). Cells below just **sync code**, **install
deps**, and **call into those scripts**.

**Before running the sync cell:** after any local agent edit you MUST commit and push
(`git add -A && git commit -m ... && git push`) so the remote kernel pulls your
latest code. Re-running the sync cell picks up new edits.

In [ ]:
!git clone https://github.com/DevaNandanJS/Benchmarking-LLM-fine-tuning-vs-training-from-scratch-using-the-same-dataset.git llm_task 2>/dev/null || (cd llm_task && git pull)
%cd llm_task

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU detected - connect a Colab GPU kernel first"
props = torch.cuda.get_device_properties(0)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(props.total_memory / 1e9, 2))
print("torch:", torch.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Phase 0 — Environment verification

The script below, `env_check.py`, is the Phase 0 environment verification — it also
serves as the "test local edit" that the sync cell is verified against (Phase 0 DoD).
Results land under `track1_finetune/logs/` and are pulled back into the repo by the user.

In [ ]:
!python track1_finetune/scripts/env_check.py

In [ ]:
!pip freeze > track1_finetune/environment.txt
print("Wrote track1_finetune/environment.txt - commit this back to the repo (reproducibility lock).")

## Phase 1 — Data Extraction & Cleaning

Runs `extract_text.py` which uses **pdfplumber** (primary) / **pypdf** (per-page fallback).

Outputs in `data/extracted/`:
- `document_clean.txt` — final cleaned text  
- `stats.json` — char / word / proxy-token counts  
- `extraction_manifest.json` — per-page extractor choice + quality scores  
- `hyphen_join_decisions.txt` — audit log of every line-break hyphen decision  
- `raw_pages/` — pre-clean text from both extractors, per page  

**After running:** open `data/extracted/document_clean.txt` and spot-check a few
random sections. Check `extraction_manifest.json` for any pages with `"garbled_flag": true`.

In [ ]:
!python track1_finetune/scripts/extract_text.py

In [ ]:
# Quick sanity check: first & last 300 chars + stats summary
with open('data/extracted/document_clean.txt', encoding='utf-8') as f:
    text = f.read()
print('--- FIRST 300 CHARS ---')
print(text[:300])
print('\n--- LAST 300 CHARS ---')
print(text[-300:])

import json
stats = json.load(open('data/extracted/stats.json'))
print('\n--- STATS ---')
for k, v in stats.items():
    if k != 'note':
        print(f'  {k}: {v}')

# Show any garbled pages
manifest = json.load(open('data/extracted/extraction_manifest.json'))
garbled = [m for m in manifest if m['garbled_flag']]
if garbled:
    print(f'\n⚠ {len(garbled)} page(s) flagged as garbled:')
    for m in garbled:
        print(f"  page {m['page']:3d}  chosen={m['extractor_chosen']}  reason={m['reason']}")
        print(f"          non_ascii={m['final_scores']['non_ascii_ratio']:.1%}  "
              f"non_dict={m['final_scores']['non_dict_word_ratio']:.1%}")
else:
    print('\n✓ No pages flagged as garbled.')